# TabPFN — Foundation Model pour données tabulaires

Test de **TabPFN v2.5** (Prior-Fitted Network) sur le sample équilibré du QRT Challenge.

TabPFN est un modèle pré-entraîné sur des données tabulaires synthétiques, qui fonctionne en quelques secondes sans hyperparameter tuning.

⚠️ **Ce notebook est prévu pour Google Colab**

In [ ]:
# Installation de TabPFN (Colab)
!pip install -q tabpfn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import time
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from tabpfn import TabPFNClassifier

SEED = 42
np.random.seed(SEED)
print('Libraries loaded.')

## 1. Monter Google Drive et charger les données

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ⚠️ ADAPTER CE CHEMIN selon ton Drive
DATA_DIR = '/content/drive/My Drive/QRT-ChallengeENS/Data'

X_raw = pd.read_csv(f'{DATA_DIR}/X_train_sample.csv')
y_raw = pd.read_csv(f'{DATA_DIR}/y_train_sample.csv')

df = X_raw.merge(y_raw, on='ROW_ID')
print(f'Dataset : {df.shape[0]:,} lignes × {df.shape[1]} colonnes')

## 2. Feature Engineering

On définit une fonction pour l'appliquer facilement au Train et au Test.

In [ ]:
def feature_engineering(df):
    data = df.copy()
    ret_cols = [f'RET_{i}' for i in range(1, 21)]
    
    # Interaction
    data['RET_1_TURNOVER'] = data['RET_1'] * data['MEDIAN_DAILY_TURNOVER']
    
    # Rolling statistics
    data['RET_MEAN_5'] = data[[f'RET_{i}' for i in range(1, 6)]].mean(axis=1)
    data['RET_MEAN_20'] = data[ret_cols].mean(axis=1)
    data['RET_STD_5'] = data[[f'RET_{i}' for i in range(1, 6)]].std(axis=1)
    data['RET_STD_20'] = data[ret_cols].std(axis=1)
    
    # Cumulative return
    data['RET_CUM_5'] = data[[f'RET_{i}' for i in range(1, 6)]].sum(axis=1)
    data['RET_CUM_20'] = data[ret_cols].sum(axis=1)
    
    # Momentum indicators
    data['RET_POSITIVE_COUNT_5'] = (data[[f'RET_{i}' for i in range(1, 6)]] > 0).sum(axis=1)
    data['RET_POSITIVE_COUNT_20'] = (data[ret_cols] > 0).sum(axis=1)
    
    # Recent vs old
    data['RET_RECENT_VS_OLD'] = data['RET_MEAN_5'] - data[[f'RET_{i}' for i in range(16, 21)]].mean(axis=1)
    
    return data

# Appliquer au train
df = feature_engineering(df)
df['target_SIGN'] = (df['target'] > 0).astype(int)

print(f'Train après FE : {df.shape}')

## 3. Préparation des données (Normalisation)

On garde les scalers en mémoire pour normaliser le test set exactement pareil.

In [ ]:
exclude_cols = ['ROW_ID', 'TS', 'ALLOCATION', 'target', 'target_SIGN']
feature_cols = [c for c in df.columns if c not in exclude_cols]

X = df[feature_cols].fillna(0).copy()
y = df['target_SIGN'].values

# Normalisation par groupe (fit)
numeric_cols = [c for c in feature_cols if c != 'GROUP']
scalers = {}

for grp in X['GROUP'].unique():
    mask = X['GROUP'] == grp
    scaler = StandardScaler()
    X.loc[mask, numeric_cols] = scaler.fit_transform(X.loc[mask, numeric_cols])
    scalers[grp] = scaler  # On sauvegarde le scaler

X = X.values.astype(np.float32)

# Split 80/20 (pour validation interne)
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

print(f'Features : {X_train.shape[1]}')
print(f'Train    : {X_train.shape[0]:,}')
print(f'Val      : {X_val.shape[0]:,}')

## 4. Entraînement sur tout le dataset (Full Train)

In [ ]:
print(f'=== TabPFN sur tout le train set ({len(X_train):,} samples) ===')
print('Cela peut prendre quelques minutes...\n')

t0 = time.time()
# ignore_pretraining_limits=True permet d'utiliser plus de données que prévu par défaut
clf_full = TabPFNClassifier(ignore_pretraining_limits=True)
clf_full.fit(X_train, y_train)

y_pred_full = clf_full.predict(X_val)
y_proba_full = clf_full.predict_proba(X_val)[:, 1]
elapsed = time.time() - t0

acc_full = accuracy_score(y_val, y_pred_full)
auc_full = roc_auc_score(y_val, y_proba_full)

print(f'Val Accuracy : {acc_full:.4f}')
print(f'Val ROC AUC  : {auc_full:.4f}')
print(f'Temps        : {elapsed:.1f}s')

## 5. Générer la Submission pour QRT

On applique le modèle entraîné (`clf_full`) sur le fichier de test (`X_test.csv`).

In [ ]:
print('=== Génération de la submission ===')

# 1. Charger le test set
X_test_raw = pd.read_csv(f'{DATA_DIR}/X_test.csv')
print(f'Test set loaded : {X_test_raw.shape}')

# 2. Feature Engineering (même fonction)
X_test_fe = feature_engineering(X_test_raw)

# 3. Préparer les features
X_test = X_test_fe[feature_cols].fillna(0).copy()
test_row_ids = X_test_fe['ROW_ID']

# 4. Normalisation (avec les scalers du train)
numeric_cols = [c for c in feature_cols if c != 'GROUP']
for grp in X_test['GROUP'].unique():
    mask = X_test['GROUP'] == grp
    if grp in scalers:
        X_test.loc[mask, numeric_cols] = scalers[grp].transform(X_test.loc[mask, numeric_cols])
    else:
        # Fallback pour nouveaux groupes (peu probable)
        scaler = StandardScaler()
        X_test.loc[mask, numeric_cols] = scaler.fit_transform(X_test.loc[mask, numeric_cols])

# 5. Conversion numpy
X_test_np = X_test.values.astype(np.float32)

# 6. Prédiction
print('Prédiction en cours...')
y_pred_test = clf_full.predict(X_test_np)

print(f'Prédictions :')
print(f'  Up (1)   : {(y_pred_test == 1).sum()} ({(y_pred_test == 1).mean()*100:.1f}%)')
print(f'  Down (0) : {(y_pred_test == 0).sum()} ({(y_pred_test == 0).mean()*100:.1f}%)')

# 7. Sauvegarde
submission = pd.DataFrame({
    'ROW_ID': test_row_ids,
    'prediction': y_pred_test
})

submission.to_csv('submission_tabpfn.csv', index=False)
print('✅ Fichier submission_tabpfn.csv créé.')

# 8. Téléchargement automatique (pour Colab)
from google.colab import files
files.download('submission_tabpfn.csv')